# MLP — Multi-Layer Perceptron

The simplest neural network architecture. Fully connected layers stacked with activations.
Everything — CNNs, RNNs, Transformers — has MLP blocks inside them.

Here we train it on a real dataset: MNIST digit classification (70,000 handwritten digits, 10 classes).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# --- Data ---
transform = transforms.Compose([
    transforms.ToTensor(),                     # PIL image → tensor [1, 28, 28], range 0-1
    transforms.Normalize((0.1307,), (0.3081,)) # normalize using MNIST mean/std
])

train_data = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

print(f'Train samples: {len(train_data)}')
print(f'Test samples:  {len(test_data)}')

# Shape of one image
X, y = next(iter(train_loader))
print(f'Batch shape: {X.shape}')   # [64, 1, 28, 28] — 64 images, 1 channel, 28×28

In [ ]:
# --- Model ---
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),           # [batch, 1, 28, 28] → [batch, 784]
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10)      # 10 output classes (digits 0-9)
            # No softmax here — CrossEntropyLoss applies it internally
        )

    def forward(self, x):
        return self.net(x)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = MLP().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Device: {device}')
print(f'Parameters: {total_params:,}')

In [ ]:
# --- Training ---
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, correct = 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (pred.argmax(1) == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)


def eval_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred  = model(X)
            total_loss += loss_fn(pred, y).item()
            correct    += (pred.argmax(1) == y).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)


for epoch in range(5):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fn, device)
    val_loss,   val_acc   = eval_epoch(model, test_loader, loss_fn, device)
    print(f'Epoch {epoch+1} | train loss: {train_loss:.3f} acc: {train_acc:.3f} | val loss: {val_loss:.3f} acc: {val_acc:.3f}')